# UniAssist — Week 1: Corpus and Textual Sources
The corpus contains 20 English source-based texts about University Metropolitan Tirana (UMT).
The source metadata records URLs and flags time-sensitive policies.
Complete the discussion questions after running the cells; generated counts are not manual gold judgments.


In [ ]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


## Task 1 — Inventory, word counts, source genres and quality


In [ ]:
rows = []
for entry, text in zip(records, texts):
    rows.append({"filename": entry["file"], "genre": entry["genre"],
                 "chars": len(text), "whitespace_tokens": len(text.split()),
                 "source_count": len(entry["sources"]),
                 "time_sensitive": entry["time_sensitive"]})
table = pd.DataFrame(rows)
display(table)
print("Total words:", table["whitespace_tokens"].sum())
print("Genres:", table["genre"].value_counts().to_dict())
print("Exact duplicate texts:", len(texts)-len(set(texts)))


## Task 2 — Tokens, types and common words using spaCy


In [ ]:
import spacy
nlp = spacy.blank("en")
all_tokens = [tok.text.lower() for text in texts for tok in nlp.make_doc(text) if not tok.is_space]
alpha = [tok for tok in all_tokens if tok.isalpha()]
print("Alphabetic tokens:", len(alpha), "types:", len(set(alpha)))
print("Type/token ratio:", round(len(set(alpha)) / max(1, len(alpha)), 4))
print("Top 15:", Counter(alpha).most_common(15))
filtered = [tok for tok in alpha if tok not in nlp.Defaults.stop_words]
print("Top 15 without stopwords:", Counter(filtered).most_common(15))


## Task 3 — UTF-8, punctuation and difficult text


In [ ]:
print("All files decoded as UTF-8")
for row, text in zip(records, texts):
    if re.search(r"[À-ÿ–—+]", text):
        print(row["file"], "has accents or special symbols")
print("Potential replacement characters:", sum(text.count("�") for text in texts))


## Task 4 — Dependency parsing and named entities
Install the model with: python -m spacy download en_core_web_sm.
Mark at least one correct and one incorrect result for each of three sentences.


In [ ]:
from spacy import displacy
try:
    trained = spacy.load("en_core_web_sm")
except OSError:
    print("Install en_core_web_sm to enable dependency and entity visualization.")
else:
    examples = [
        "University Metropolitan Tirana has a Faculty of Computer Science and IT.",
        "The university's English admissions page discusses State Matura.",
        "Erasmus+ students may need a Learning Agreement."
    ]
    for sentence in examples:
        doc = trained(sentence)
        print(sentence, [(e.text, e.label_) for e in doc.ents])
        from IPython.display import HTML
        display(HTML(displacy.render(doc, style="dep", jupyter=False)))
        display(HTML(displacy.render(doc, style="ent", jupyter=False)))


## Submit
1. Three sentences describing the corpus, its source and its genres.
2. Corpus statistics and 10 most frequent words.
3. Three analyzed sentences with one correct and one incorrect NLP annotation each.
4. Source list and copyright/permission notes; explain why outdated rules are flagged.
